## Integer Damath DynaQ

In [14]:
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any
import numpy as np
import copy
from collections import defaultdict, deque


In [15]:
Operator = Optional[str]  # '+', '-', 'x', '/' or None

@dataclass
class Piece:
    player: int  # 1 = Blue, -1 = Red
    value: int
    dama: bool = False

    def __repr__(self):
        # Convert B to blue emoji and R to red emoji for better visualization
        p = "🔵" if self.player == 1 else "🔴"
        val = f"{self.value:+d}" if self.value >= 0 else str(self.value)
        return f"{p}{'D' if self.dama else ''}{val}"
    
    
    def copy(self):
        return Piece(self.player, self.value, self.dama)

@dataclass
class Move:
    path: List[Tuple[int, int]]            # sequence of positions traversed
    captures: List[Tuple[int, int]]        # list of captured piece positions
    promotes: bool = False                 # whether the move results in promotion
    score_gain: int = 0                    # arithmetic reward from the move
    is_dama_capture: bool = False          # whether move made by dama
    is_multi_jump: bool = False            # whether multiple captures occurred

    def __repr__(self):
        cap_str = f" x{len(self.captures)}" if self.captures else ""
        promo = " (promo)" if self.promotes else ""
        return f"{self.path}{cap_str}{promo} +{self.score_gain}"



In [16]:
class DamathEnv:
    def __init__(self, rows=8, cols=8, operator_pattern=None):
        self.R = rows
        self.C = cols
        assert rows==8 and cols==8, "Currently implemented for 8x8 boards."
        # Operators on playable squares. Default pattern similar to provided image if None.
        if operator_pattern is None:
            operator_pattern = self.default_operator_board()
        self.op_board = operator_pattern
        # Piece board: dict (r,c)->Piece
        self.pieces: Dict[Tuple[int,int], Piece] = {}
        # Scores cumulative per player
        self.scores = {1: 0.0, -1: 0.0}
        self.to_move = 1  # 1 starts (blue on top)
        self.history_states = deque(maxlen=50)  # for repetition detection (store simple board hashes)
        
        # initialize sample starting board if user wants. We'll provide a helper to set initial config.
        self.init_default_integer_setup()

    def default_operator_board(self):
        # Create operator layout (8x8) using a repeating pattern similar to the uploaded assets.
        # Operators placed on playable squares (r+c)%2==1.
        ops = ['x','/','-','+']  # cycle
        board = [[None for _ in range(self.C)] for __ in range(self.R)]
        for r in range(self.R):
            for c in range(self.C):
                if (r + c) % 2 == 1:
                    # choose operator based on some pattern; rotate every cell
                    board[r][c] = ops[(r + 2*c) % len(ops)]
                else:
                    board[r][c] = None
        return board

    def init_default_integer_setup(self):
        # Initialize pieces according to the "integer damath" sample. We'll follow a symmetric-ish layout.
        # Blue (player=1) on top three rows playable squares, Red (player=-1) on bottom three rows.
        self.pieces = {}
        # sample integer values, you can customize to exact image mapping
        blue_values = [
            [-11, 8, -5, 2],
            [0, -3, 10, -7],
            [-9, 6, -1, 4],
        ]
        red_values = [
            [4, -1, 6, -9],
            [-7, 10, -3, 0],
            [2, -5, 8, -11]
        ]
        # place on playable squares; for top rows choose columns 0,2,4,6 for row 0,1.. pattern.
        # We'll place blues on rows 0..2 and reds on rows 5..7 so that they face each other.
        # map values left to right
        def playable_positions_on_row(r):
            # playable cols where (r+c)%2==1
            return [c for c in range(self.C) if (r+c)%2==1]
        # Blue top 3 rows
        for i, r in enumerate(range(0,3)):
            cols = playable_positions_on_row(r)
            vals = blue_values[i]
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=1, value=vals[j], dama=False)
        # Red bottom 3 rows
        for i, r in enumerate(range(5,8)):
            cols = playable_positions_on_row(r)
            vals = red_values[i-0] if i < len(red_values) else []
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=-1, value=vals[j], dama=False)
        # reset scores and to_move
        self.scores = {1:0.0, -1:0.0}
        self.to_move = 1
        self.history_states.clear()
        self.record_state()

    def copy(self):
        newenv = DamathEnv(self.R, self.C)
        newenv.op_board = copy.deepcopy(self.op_board)
        newenv.pieces = {k: v.copy() for k,v in self.pieces.items()}
        newenv.scores = dict(self.scores)
        newenv.to_move = self.to_move
        newenv.history_states = copy.deepcopy(self.history_states)
        return newenv

    def in_bounds(self, r,c):
        return 0 <= r < self.R and 0 <= c < self.C

    def is_playable(self, r,c):
        return self.in_bounds(r,c) and ((r+c)%2==1)

    def get_piece(self, r,c) -> Optional[Piece]:
        return self.pieces.get((r,c))

    def remove_piece(self, r,c):
        if (r,c) in self.pieces:
            del self.pieces[(r,c)]

    def move_piece(self, from_rc, to_rc):
        p = self.pieces.pop(from_rc)
        self.pieces[to_rc] = p
        return p

    def record_state(self):
        # Simple hash of pieces positions and values and to_move for repetition detection
        items = tuple(sorted([ (pos, piece.player, piece.value, piece.dama) for pos,piece in self.pieces.items() ]))
        key = (self.to_move, items)
        self.history_states.append(key)

    # ----------------------------- Operators & arithmetic -----------------------------
    def op_at(self, r,c):
        if not self.is_playable(r,c):
            return None
        return self.op_board[r][c]

    def apply_operator(self, op: str, a: int, b: int):
        if op == '+':
            return a + b
        if op == '-':
            return a - b
        if op == 'x' or op == 'X' or op == '*':
            return a * b
        if op == '/':
            # integer division semantics: handle division by zero and prefer integer division rounding toward zero
            if b == 0:
                # define a penalty or large negative? For now, return 0 to avoid crash.
                return 0
            return int(a / b)
        raise ValueError("Unknown op "+str(op))

    # ----------------------------- Movement and capture generation -----------------------------
    def generate_all_moves(self, player:int):
        """
        Returns a list of Move objects representing all legal moves for player.
        Enforces mandatory captures and priority rules described in prompt.
        """
        # 1) Find all capture sequences for every piece
        capture_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            caps = self._generate_captures_from(pos, piece)
            capture_moves.extend(caps)
        if len(capture_moves) > 0:
            # enforce capture priority: (1) max captures, (2) if tie, dama priority, (3) if regular has more captures than dama, regular wins
            max_cap = max(len(m.captures) for m in capture_moves)
            # filter moves with max captures
            maxcap_moves = [m for m in capture_moves if len(m.captures)==max_cap]
            # among tied moves, if any are from dama pieces and any from regular, prefer dama (unless regular has strictly more captures in other moves)
            # but since we already filtered to max_cap, only need to prefer dama among ties => prefer moves with piece.dama True if any
            if any(self.pieces[m.path[0]].dama for m in maxcap_moves): # note m.path[0] original square
                maxcap_moves = [m for m in maxcap_moves if self.pieces[m.path[0]].dama]
            # compute score_gain for each move using current op board and multipliers
            for m in maxcap_moves:
                m.score_gain = self._compute_move_score(m, player)
            return maxcap_moves
        # 2) If no captures, generate simple moves including dama moves
        simple_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            sms = self._generate_simple_from(pos, piece)
            simple_moves.extend(sms)
        # mark score_gain zero for these
        for m in simple_moves:
            m.score_gain = 0.0
        return simple_moves

    def _generate_simple_from(self, pos, piece:Piece):
        r,c = pos
        moves = []
        if piece.dama:
            # dama can move any distance along diagonals (like king in international draughts)
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                step=1
                while True:
                    nr = r + dr*step; nc = c + dc*step
                    if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): break
                    if (nr,nc) in self.pieces: break
                    path = [(r,c),(nr,nc)]
                    promotes = self._check_promotion(nr, piece.player)
                    moves.append(Move(path=path, captures=[], promotes=promotes))
                    step += 1
        else:
            # regular piece: forward-only? In Damath, regular pieces can move diagonally forward one space.
            # We assume player=1 moves 'down' (increasing row), player=-1 moves 'up' (decreasing row).
            dr = 1 if piece.player==1 else -1
            for dc in (-1,1):
                nr = r + dr; nc = c + dc
                if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): continue
                if (nr,nc) in self.pieces: continue
                path=[(r,c),(nr,nc)]
                promotes = self._check_promotion(nr, piece.player)
                moves.append(Move(path=path, captures=[], promotes=promotes))
        return moves

    def _generate_captures_from(self, pos, piece:Piece):
        # returns all capture sequences starting from this piece (as Move objects)
        # For regular pieces: jump over adjacent opponent piece landing on square beyond if empty; can chain.
        # For dama pieces: long-range capture along diagonals: can jump over an opponent piece that has at least one empty landing square beyond it on the same diagonal. Dama can land on any empty square beyond the captured piece on that diagonal (but rules about priority when multiple captures available are handled globally).
        results = []
        r,c = pos

        if piece.dama:
            # long-range captures: for each diagonal, find opponent pieces and possible landing squares beyond.
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                # step along diagonal to find first opponent piece(s)
                step=1
                while True:
                    mr = r + dr*step; mc = c + dc*step
                    if not self.in_bounds(mr,mc) or not self.is_playable(mr,mc): break
                    if (mr,mc) in self.pieces:
                        target = self.pieces[(mr,mc)]
                        if target.player == piece.player:
                            break  # blocked by own piece
                        # find landing squares beyond (must be empty)
                        land_step = 1
                        while True:
                            lr = mr + dr*land_step; lc = mc + dc*land_step
                            if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): break
                            if (lr,lc) in self.pieces: break
                            # found a possible landing square
                            # create a tentative move: capture that one piece and land on (lr,lc)
                            new_env = self.copy()
                            # perform capture on new_env to continue searching for multi-captures
                            captured_piece = new_env.pieces.pop((mr,mc))
                            moved_piece = new_env.pieces.pop((r,c))
                            new_env.pieces[(lr,lc)] = moved_piece
                            # Recurse to find further captures from (lr,lc)
                            # Note: we store the captured piece object snapshot as part of capture tuple
                            further = new_env._generate_captures_from((lr,lc), moved_piece)
                            if len(further)==0:
                                m = Move(path=[(r,c),(lr,lc)], captures=[(mr,mc,captured_piece)], promotes=False)
                                results.append(m)
                            else:
                                for fm in further:
                                    # prepend current capture to fm
                                    m = Move(path=[(r,c)] + fm.path, captures=[(mr,mc,captured_piece)] + fm.captures, promotes=False)
                                    results.append(m)
                            land_step += 1
                        break  # only consider the first opponent piece along diagonal for long-range capture
                    else:
                        step += 1

        else:
            # regular piece captures: check adjacent diagonals for opponent piece and landing square beyond
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                ar = r + dr; ac = c + dc  # adjacent
                lr = r + 2*dr; lc = c + 2*dc  # landing beyond
                if not self.in_bounds(ar,ac) or not self.is_playable(ar,ac): continue
                if (ar,ac) not in self.pieces: continue
                if self.pieces[(ar,ac)].player == piece.player: continue
                if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): continue
                if (lr,lc) in self.pieces: continue
                # simulate capture and recurse
                new_env = self.copy()
                captured_piece = new_env.pieces.pop((ar,ac))
                moved_piece = new_env.pieces.pop((r,c))
                new_env.pieces[(lr,lc)] = moved_piece
                further = new_env._generate_captures_from((lr,lc), moved_piece)
                if len(further)==0:
                    m = Move(path=[(r,c),(lr,lc)], captures=[(ar,ac,captured_piece)], promotes=self._check_promotion(lr,piece.player))
                    results.append(m)
                else:
                    for fm in further:
                        m = Move(path=[(r,c)] + fm.path, captures=[(ar,ac,captured_piece)] + fm.captures, promotes=self._check_promotion(fm.path[-1][0], piece.player))
                        results.append(m)
        # remove duplicate sequences (same path) - dedupe by path and captured coordinates
        uniq = {}
        for m in results:
            key = (tuple(m.path), tuple((r,c,cp.value) for r,c,cp in m.captures))
            if key not in uniq or len(uniq[key].captures) < len(m.captures):
                uniq[key] = m
        return list(uniq.values())

    def _compute_move_score(self, move:Move, mover_player:int):
        # compute arithmetic score gain to mover for a capture move following rules:
        # - each captured piece adds op(own_value, captured_value) where op is operator on the landing square of that jump
        # - for dama captures, double score; if both dama, quadruple for that take.
        # - if a dama is taken by regular, the score is doubled as well (we handle multiplier on capture event)
        total = 0.0
        # need to reconstruct the piece value used in each jump: the moving piece's value at that time can be assumed unchanged
        # We'll use the mover's piece's original value
        # For multi-jumps, landing squares determine operators used
        mover_start = move.path[0]
        mover_piece = self.pieces.get(mover_start)
        mover_is_dama = mover_piece.dama if mover_piece else False
        # We must approximate the moving piece's value; in rules it's the mover's own piece value each time
        mover_value = mover_piece.value if mover_piece else 0
        # For each capture in sequence, determine operator at landing square (the square after the specific jump)
        # To find landing square for ith capture: it's path[1+i]
        for i, (cap_r,cap_c,cap_piece) in enumerate(move.captures):
            landing = move.path[1+i] if len(move.path) > 1+i else move.path[-1]
            op = self.op_at(*landing)
            # apply operator: op(self_value, captured_value)
            base = self.apply_operator(op, mover_value, cap_piece.value)
            # multipliers:
            mult = 1
            # If mover is dama, double for that take; if captured is dama and mover is dama -> quadruple (2*2)
            if mover_is_dama and cap_piece.dama:
                mult = 4
            elif mover_is_dama:
                mult = 2
            elif cap_piece.dama:
                # captured is dama and mover is regular: score doubled as well per rules
                mult = 2
            total += base * mult
        return total

    def _check_promotion(self, r, player):
        # If a piece reaches opposing end (row 7 for player=1, row 0 for player=-1), it is promoted
        if player==1 and r==self.R-1: return True
        if player==-1 and r==0: return True
        return False

    def apply_move(self, move: Move, verbose=False):
        """
        Apply a move and compute rewards focusing ONLY on:
        - Arithmetic Gain (from captures)
        - Dama Promotion Bonus
        - Terminal Result (win/loss outcome)
        """

        player = self.to_move
        total_gain = 0.0
        dama_bonus = 0.0

        # ----- 1. Simple move -----
        if len(move.captures) == 0:
            frm, to = move.path[0], move.path[-1]
            piece = self.pieces.pop(frm)
            self.pieces[to] = piece

            # Promotion check
            if self._check_promotion(to[0], piece.player) and not piece.dama:
                piece.dama = True
                dama_bonus = 0.4  # reward for becoming dama

        # ----- 2. Capture move -----
        else:
            frm = move.path[0]
            mover = self.pieces.pop(frm)
            current_pos = frm

            for i, (cap_r, cap_c, cap_piece_snapshot) in enumerate(move.captures):
                landing = move.path[1 + i] if 1 + i < len(move.path) else move.path[-1]
                op = self.op_at(*landing)
                captured_piece = self.pieces.pop((cap_r, cap_c))

                # Arithmetic gain
                base = self.apply_operator(op, mover.value, captured_piece.value)

                # Dama-related multipliers
                mult = 1
                if mover.dama and captured_piece.dama:
                    mult = 4
                elif mover.dama or captured_piece.dama:
                    mult = 2

                gain = base * mult
                total_gain += gain
                current_pos = landing

            # Place mover at final position
            self.pieces[current_pos] = mover

            # Promotion check
            if self._check_promotion(current_pos[0], mover.player) and not mover.dama:
                mover.dama = True
                dama_bonus = 0.4

            # Update cumulative score
            self.scores[mover.player] += total_gain

        # ----- 3. Compute immediate reward -----
        # Normalize capture gain to avoid exploding gradients in RL
        normalized_gain = np.tanh(total_gain / 10.0)
        total_reward = 0.8 * normalized_gain + dama_bonus
        total_reward = float(np.clip(total_reward, -1.0, 1.0))

        # ----- 4. Switch turn and record -----
        self.to_move *= -1
        self.record_state()

        # ----- 5. Terminal check -----
        if self.game_over():
            final_scores, winner = self.final_scores_and_winner()
            score_diff = (final_scores[player] - final_scores[-player]) / 100.0

            if winner == player:
                total_reward += 1.0 + np.tanh(score_diff)
            elif winner == -player:
                total_reward -= 1.0 + np.tanh(abs(score_diff))
            total_reward = float(np.clip(total_reward, -1.0, 1.0))

        # ----- 6. Verbose log -----
        if verbose:
            print(
                f"[Player {player}] gain={total_gain:.2f}, reward={total_reward:.2f}, "
                f"dama={dama_bonus:.2f}, scores={self.scores}"
            )

        return total_reward




    
    def _board_advantage(self):
        """Compute normalized material advantage of current board."""
        blue_value = sum(p.value * (2 if p.dama else 1) for p in self.pieces.values() if p.player == 1)
        red_value  = sum(p.value * (2 if p.dama else 1) for p in self.pieces.values() if p.player == -1)
        return (blue_value - red_value) / 100.0


    # ----------------------------- Game end and scoring -----------------------------
    def legal_moves_exist(self, player:int):
        return len(self.generate_all_moves(player))>0

    def game_over(self):
        # game over if current player to move has no moves or only one player's chips remain or repetition or stalemate
        if not self.legal_moves_exist(self.to_move):
            return True
        # if only chips of one player remain
        players_present = set(p.player for p in self.pieces.values())
        if len(players_present) <= 1:
            return True
        # repetition detection (simple): if last 6 states repeated pattern
        # For now, consider repetition if history has same state repeated >=4 times overall
        hist = list(self.history_states)
        if len(hist) >= 8:
            counts = defaultdict(int)
            for h in hist:
                counts[h] += 1
                if counts[h] >= 4:
                    return True
        return False

    def final_scores_and_winner(self):
        # Add remaining pieces to their player's cumulative scores (dama doubled)
        final_scores = dict(self.scores)
        for (r,c), piece in self.pieces.items():
            val = piece.value * (2 if piece.dama else 1)
            final_scores[piece.player] += val
        # Determine winner
        if final_scores[1] > final_scores[-1]:
            winner = 1
        elif final_scores[1] < final_scores[-1]:
            winner = -1
        else:
            winner = 0
        return final_scores, winner

    # ----------------------------- Utilities -----------------------------
    def print_board(self):
        # Create board grid with operators and pieces
        grid = [[" ." for _ in range(self.C)] for __ in range(self.R)]
        for y in range(self.R):
            for x in range(self.C):
                if not self.is_playable(y, x):
                    grid[y][x] = "##"
                else:
                    op = self.op_at(y, x)
                    grid[y][x] = f" {op}"
        # Place pieces
        for (y, x), piece in self.pieces.items():
            sym = '🔵' if piece.player == 1 else '🔴'
            if piece.dama:
                sym += 'K'
            grid[y][x] = f"{sym}{piece.value:02d}" if piece.value >= 0 else f"{sym}{piece.value}"

        # Print column headers
        print("\n     " + " ".join([f"{x:>4}" for x in range(self.C)]))
        print("     " + "----" * self.C)

        # Print from top (highest y) to bottom (y=0)
        for y in reversed(range(self.R)):
            row_str = " ".join(f"{cell:>4}" for cell in grid[y])
            print(f"{y:>2} | {row_str} | {y:>2}")

        print("     " + "----" * self.C)
        print("     " + " ".join([f"{x:>4}" for x in range(self.C)]))


In [17]:
# Operator layout: (y, x) format
# y=0 is bottom row, y=7 is top row

operator_pattern_official = [
    ['x', '-', '/', 'x', '-', '+', '+', 'x'],
    ['-', '/', '-', 'x', '-', '+', 'x', '-'],
    ['-', '+', '+', '+', 'x', 'x', '/', '+'],
    ['x', '+', '+', '-', 'x', '/', '+', 'x'],
    ['x', '-', '/', 'x', '-', '-', '+', 'x'],
    ['-', '/', 'x', 'x', '-', '+', 'x', '-'],
    ['-', 'x', '+', '+', 'x', 'x', '/', '+'],
    ['+', '/', '-', '-', 'x', '/', '+', 'x']
]

env = DamathEnv(rows = 8, cols = 8, operator_pattern=operator_pattern_official)
env.print_board()
moves = env.generate_all_moves(env.to_move)
print("\nAvailable moves for player", env.to_move, "->", len(moves))
for move in moves:
    start = move.path[0]
    end = move.path[-1]
    piece = env.pieces.get(start, None)
    if piece:
        piece_type = "Dama" if piece.dama else "Regular"
        # Swap (y, x) to (x, y) for readability
        print(f"{piece_type} {piece.value:+d} at ({start[1]}, {start[0]}) -> ({end[1]}, {end[0]}) | ΔScore: {move.score_gain:+.1f}")
    else:
        print(f"Unknown piece at ({start[1]}, {start[0]}) -> ({end[1]}, {end[0]}) | ΔScore: {move.score_gain:+.1f}")




        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    -   ##    x   ##    -   ##    x |  4
 3 |    x   ##    +   ##    x   ##    +   ## |  3
 2 |   ##  🔵-9   ##  🔵06   ##  🔵-1   ##  🔵04 |  2
 1 |  🔵00   ##  🔵-3   ##  🔵10   ##  🔵-7   ## |  1
 0 |   ## 🔵-11   ##  🔵08   ##  🔵-5   ##  🔵02 |  0
     --------------------------------
        0    1    2    3    4    5    6    7

Available moves for player 1 -> 7
Regular -9 at (1, 2) -> (0, 3) | ΔScore: +0.0
Regular -9 at (1, 2) -> (2, 3) | ΔScore: +0.0
Regular +6 at (3, 2) -> (2, 3) | ΔScore: +0.0
Regular +6 at (3, 2) -> (4, 3) | ΔScore: +0.0
Regular -1 at (5, 2) -> (4, 3) | ΔScore: +0.0
Regular -1 at (5, 2) -> (6, 3) | ΔScore: +0.0
Regular +4 at (7, 2) -> (6, 3) | ΔScore: +0.0


In [18]:
operator_pattern_official = [
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '/', '/', '-', '-', '/', '+', 'x']
]

In [19]:
# Print empty board with no pieces, just operators
env_empty = DamathEnv(rows = 8, cols = 8, operator_pattern=operator_pattern_official)
env_empty.pieces = {}  # clear pieces
env_empty.print_board()


        0    1    2    3    4    5    6    7
     --------------------------------
 7 |    x   ##    /   ##    -   ##    +   ## |  7
 6 |   ##    /   ##    x   ##    +   ##    - |  6
 5 |    -   ##    +   ##    x   ##    /   ## |  5
 4 |   ##    +   ##    -   ##    /   ##    x |  4
 3 |    x   ##    /   ##    -   ##    +   ## |  3
 2 |   ##    /   ##    x   ##    +   ##    - |  2
 1 |    -   ##    +   ##    x   ##    /   ## |  1
 0 |   ##    +   ##    -   ##    /   ##    x |  0
     --------------------------------
        0    1    2    3    4    5    6    7


In [20]:
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# create a unique run folder
run_name = f"runs/damath_selfplay_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(run_name)

In [21]:
import numpy as np
import random
import pickle
import hashlib
from collections import defaultdict, deque
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

# ----------------------- State-Action Representation for Dyna-Q ------------------------
def state_to_hash(env):
    """Convert environment state to a hashable representation for Q-table indexing"""
    state_items = []
    for pos in sorted(env.pieces.keys()):
        piece = env.pieces[pos]
        state_items.append((pos, piece.player, piece.value, piece.dama))
    
    state_tuple = (env.to_move, tuple(state_items))
    return hashlib.md5(str(state_tuple).encode()).hexdigest()[:16]

def move_to_action_id(move):
    """Convert a move to a unique action identifier"""
    start = move.path[0]
    end = move.path[-1]
    captures = len(move.captures)
    return f"{start[0]},{start[1]}->{end[0]},{end[1]}|cap:{captures}"

def get_legal_actions(env):
    """Get all legal moves as action IDs"""
    moves = env.generate_all_moves(env.to_move)
    return [move_to_action_id(move) for move in moves], moves

def get_legal_actions_from_state(env):
    """Helper function to get legal actions from a state"""
    moves = env.generate_all_moves(env.to_move)
    return [move_to_action_id(move) for move in moves], moves

# # ----------------------- Dyna-Q Agent ------------------------
# class DynaQAgent:
#     def __init__(self, alpha=0.1, gamma=0.95, epsilon=0.1, planning_steps=50):
#         self.alpha = alpha  # learning rate
#         self.gamma = gamma  # discount factor
#         self.epsilon = epsilon  # exploration rate
#         self.planning_steps = planning_steps  # number of planning steps per real step
        
#         # Q-table: Q[state][action] = value
#         self.Q = defaultdict(lambda: defaultdict(float))
        
#         # Model: model[state][action] = (next_state, reward)
#         self.model = defaultdict(lambda: defaultdict(lambda: (None, 0.0)))
        
#         # Track visited state-action pairs for planning
#         self.visited_sa = set()
        
#         # Experience replay buffer for more stable learning
#         self.experience_buffer = deque(maxlen=10000)
        
#         # Reward normalization statistics
#         self.reward_history = deque(maxlen=1000)
#         self.running_mean = 0.0
#         self.running_std = 1.0
        
#     def normalize_reward(self, raw_reward):
#         """Normalize reward using running statistics"""
#         self.reward_history.append(raw_reward)
        
#         # Update running statistics
#         if len(self.reward_history) > 10:
#             self.running_mean = np.mean(self.reward_history)
#             self.running_std = np.std(self.reward_history) + 1e-8  # Avoid division by zero
        
#         # Z-score normalization
#         normalized = (raw_reward - self.running_mean) / self.running_std
        
#         # Clip to prevent extreme values
#         return np.clip(normalized, -5.0, 5.0)
        
#     def get_action(self, env, legal_moves, training=True):
#         """Select action using epsilon-greedy policy"""
#         state_hash = state_to_hash(env)
#         action_ids, moves = get_legal_actions(env)
        
#         if not action_ids:
#             return None, None
            
#         if training and random.random() < self.epsilon:
#             # Random exploration
#             chosen_action = random.choice(action_ids)
#         else:
#             # Greedy action selection
#             q_values = [self.Q[state_hash][action] for action in action_ids]
#             max_q = max(q_values)
#             best_actions = [action_ids[i] for i, q in enumerate(q_values) if q == max_q]
#             chosen_action = random.choice(best_actions)
        
#         # Find corresponding move
#         chosen_move = None
#         for i, action_id in enumerate(action_ids):
#             if action_id == chosen_action:
#                 chosen_move = moves[i]
#                 break
                
#         return chosen_move, chosen_action
    
#     def update_q_value(self, state, action, reward, next_state, done):
#         """Update Q-value using Q-learning update rule"""
#         state_hash = state_to_hash(state)
        
#         if done:
#             td_target = reward
#         else:
#             # Get maximum Q-value for next state
#             next_actions, _ = get_legal_actions_from_state(next_state)
#             if next_actions:
#                 next_state_hash = state_to_hash(next_state)
#                 max_next_q = max([self.Q[next_state_hash][a] for a in next_actions])
#             else:
#                 max_next_q = 0.0
#             td_target = reward + self.gamma * max_next_q
        
#         current_q = self.Q[state_hash][action]
#         self.Q[state_hash][action] = current_q + self.alpha * (td_target - current_q)
    
#     def update_model(self, state, action, next_state, reward):
#         """Update the environment model"""
#         state_hash = state_to_hash(state)
#         next_state_hash = state_to_hash(next_state) if next_state else None
#         self.model[state_hash][action] = (next_state_hash, reward)
#         self.visited_sa.add((state_hash, action))
    
#     def plan(self):
#         """Perform planning steps using the learned model"""
#         if len(self.visited_sa) < self.planning_steps:
#             planning_steps = len(self.visited_sa)
#         else:
#             planning_steps = self.planning_steps
            
#         for _ in range(planning_steps):
#             if not self.visited_sa:
#                 break
                
#             # Randomly sample a visited state-action pair
#             state_hash, action = random.choice(list(self.visited_sa))
            
#             # Get predicted next state and reward from model
#             next_state_hash, reward = self.model[state_hash][action]
            
#             if next_state_hash is not None:
#                 # Simulate Q-learning update
#                 next_actions = list(self.Q[next_state_hash].keys())
#                 if next_actions:
#                     max_next_q = max([self.Q[next_state_hash][a] for a in next_actions])
#                 else:
#                     max_next_q = 0.0
                    
#                 td_target = reward + self.gamma * max_next_q
#                 current_q = self.Q[state_hash][action]
#                 self.Q[state_hash][action] = current_q + self.alpha * (td_target - current_q)
    
#     def save_agent(self, filepath):
#         """Save the agent's Q-table and model"""
#         agent_data = {
#             'Q': dict(self.Q),
#             'model': dict(self.model),
#             'visited_sa': list(self.visited_sa),
#             'alpha': self.alpha,
#             'gamma': self.gamma,
#             'epsilon': self.epsilon,
#             'planning_steps': self.planning_steps,
#             'running_mean': self.running_mean,
#             'running_std': self.running_std,
#             'reward_history': list(self.reward_history)
#         }
#         with open(filepath, 'wb') as f:
#             pickle.dump(agent_data, f)
    
#     def load_agent(self, filepath):
#         """Load the agent's Q-table and model"""
#         with open(filepath, 'rb') as f:
#             agent_data = pickle.load(f)
        
#         self.Q = defaultdict(lambda: defaultdict(float), agent_data['Q'])
#         self.model = defaultdict(lambda: defaultdict(lambda: (None, 0.0)), agent_data['model'])
#         self.visited_sa = set(agent_data['visited_sa'])
#         self.alpha = agent_data['alpha']
#         self.gamma = agent_data['gamma']
#         self.epsilon = agent_data['epsilon']
#         self.planning_steps = agent_data['planning_steps']
#         self.running_mean = agent_data.get('running_mean', 0.0)
#         self.running_std = agent_data.get('running_std', 1.0)
#         self.reward_history = deque(agent_data.get('reward_history', []), maxlen=1000)

# ----------------------- Dyna-Q Agent with Fixed Serialization ------------------------
class DynaQAgent:
    def __init__(self, alpha=0.1, gamma=0.95, epsilon=0.1, planning_steps=50):
        self.alpha = alpha  # learning rate
        self.gamma = gamma  # discount factor
        self.epsilon = epsilon  # exploration rate
        self.planning_steps = planning_steps  # number of planning steps per real step
        
        # Q-table: Q[state][action] = value
        self.Q = defaultdict(lambda: defaultdict(float))
        
        # Model: model[state][action] = (next_state, reward)
        self.model = defaultdict(lambda: defaultdict(lambda: (None, 0.0)))
        
        # Track visited state-action pairs for planning
        self.visited_sa = set()
        
        # Experience replay buffer for more stable learning
        self.experience_buffer = deque(maxlen=10000)
        
        # Reward normalization statistics
        self.reward_history = deque(maxlen=1000)
        self.running_mean = 0.0
        self.running_std = 1.0
        
    def normalize_reward(self, raw_reward):
        """Normalize reward using running statistics"""
        self.reward_history.append(raw_reward)
        
        # Update running statistics
        if len(self.reward_history) > 10:
            self.running_mean = np.mean(self.reward_history)
            self.running_std = np.std(self.reward_history) + 1e-8  # Avoid division by zero
        
        # Z-score normalization
        normalized = (raw_reward - self.running_mean) / self.running_std
        
        # Clip to prevent extreme values
        return np.clip(normalized, -1.0, 1.0)
        
    def get_action(self, env, legal_moves, training=True):
        """Select action using epsilon-greedy policy"""
        state_hash = state_to_hash(env)
        action_ids, moves = get_legal_actions(env)
        
        if not action_ids:
            return None, None
            
        if training and random.random() < self.epsilon:
            # Random exploration
            chosen_action = random.choice(action_ids)
        else:
            # Greedy action selection
            q_values = [self.Q[state_hash][action] for action in action_ids]
            max_q = max(q_values)
            best_actions = [action_ids[i] for i, q in enumerate(q_values) if q == max_q]
            chosen_action = random.choice(best_actions)
        
        # Find corresponding move
        chosen_move = None
        for i, action_id in enumerate(action_ids):
            if action_id == chosen_action:
                chosen_move = moves[i]
                break
                
        return chosen_move, chosen_action
    
    def update_q_value(self, state, action, reward, next_state, done):
        """Update Q-value using Q-learning update rule"""
        state_hash = state_to_hash(state)
        
        if done:
            td_target = reward
        else:
            # Get maximum Q-value for next state
            next_actions, _ = get_legal_actions_from_state(next_state)
            if next_actions:
                next_state_hash = state_to_hash(next_state)
                max_next_q = max([self.Q[next_state_hash][a] for a in next_actions])
            else:
                max_next_q = 0.0
            td_target = reward + self.gamma * max_next_q
        
        current_q = self.Q[state_hash][action]
        self.Q[state_hash][action] = current_q + self.alpha * (td_target - current_q)
    
    def update_model(self, state, action, next_state, reward):
        """Update the environment model"""
        state_hash = state_to_hash(state)
        next_state_hash = state_to_hash(next_state) if next_state else None
        self.model[state_hash][action] = (next_state_hash, reward)
        self.visited_sa.add((state_hash, action))
    
    def plan(self):
        """Perform planning steps using the learned model"""
        if len(self.visited_sa) < self.planning_steps:
            planning_steps = len(self.visited_sa)
        else:
            planning_steps = self.planning_steps
            
        for _ in range(planning_steps):
            if not self.visited_sa:
                break
                
            # Randomly sample a visited state-action pair
            state_hash, action = random.choice(list(self.visited_sa))
            
            # Get predicted next state and reward from model
            next_state_hash, reward = self.model[state_hash][action]
            
            if next_state_hash is not None:
                # Simulate Q-learning update
                next_actions = list(self.Q[next_state_hash].keys())
                if next_actions:
                    max_next_q = max([self.Q[next_state_hash][a] for a in next_actions])
                else:
                    max_next_q = 0.0
                    
                td_target = reward + self.gamma * max_next_q
                current_q = self.Q[state_hash][action]
                self.Q[state_hash][action] = current_q + self.alpha * (td_target - current_q)
    
    def save_agent(self, filepath):
        """Save the agent's Q-table and model - converts defaultdicts to regular dicts"""
        # Convert nested defaultdicts to regular nested dicts for pickle
        q_dict = {}
        for state, actions in self.Q.items():
            q_dict[state] = dict(actions)
        
        model_dict = {}
        for state, actions in self.model.items():
            model_dict[state] = dict(actions)
        
        agent_data = {
            'Q': q_dict,
            'model': model_dict,
            'visited_sa': list(self.visited_sa),
            'alpha': self.alpha,
            'gamma': self.gamma,
            'epsilon': self.epsilon,
            'planning_steps': self.planning_steps,
            'running_mean': self.running_mean,
            'running_std': self.running_std,
            'reward_history': list(self.reward_history)
        }
        
        with open(filepath, 'wb') as f:
            pickle.dump(agent_data, f)
        
        print(f"✅ Agent saved successfully to {filepath}")
    
    def load_agent(self, filepath):
        """Load the agent's Q-table and model"""
        with open(filepath, 'rb') as f:
            agent_data = pickle.load(f)
        
        # Reconstruct defaultdicts from regular dicts
        self.Q = defaultdict(lambda: defaultdict(float))
        for state, actions in agent_data['Q'].items():
            for action, value in actions.items():
                self.Q[state][action] = value
        
        self.model = defaultdict(lambda: defaultdict(lambda: (None, 0.0)))
        for state, actions in agent_data['model'].items():
            for action, value in actions.items():
                self.model[state][action] = value
        
        self.visited_sa = set(agent_data['visited_sa'])
        self.alpha = agent_data['alpha']
        self.gamma = agent_data['gamma']
        self.epsilon = agent_data['epsilon']
        self.planning_steps = agent_data['planning_steps']
        self.running_mean = agent_data.get('running_mean', 0.0)
        self.running_std = agent_data.get('running_std', 1.0)
        self.reward_history = deque(agent_data.get('reward_history', []), maxlen=1000)
        
        print(f"✅ Agent loaded successfully from {filepath}")
        print(f"   Q-table size: {len(self.Q)} states")
        print(f"   Model size: {len(self.visited_sa)} state-action pairs")
# ----------------------- Dyna-Q Training Function with Normalized Rewards ------------------------
def train_dyna_q_agent(env_factory, agent, num_episodes=1000, max_moves_per_game=300):
    """Train the Dyna-Q agent through self-play with normalized rewards"""
    
    episode_rewards = []
    episode_lengths = []
    win_counts = {1: 0, -1: 0, 0: 0}
    
    # TensorBoard logging
    run_name = f"runs/damath_dynaq_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    writer = SummaryWriter(run_name)
    print(f"📈 Logging Dyna-Q training to TensorBoard: {run_name}")
    
    for episode in range(1, num_episodes + 1):
        env = env_factory()
        total_normalized_reward = 0.0
        total_raw_reward = 0.0
        move_count = 0
        
        while not env.game_over() and move_count < max_moves_per_game:
            current_state = env.copy()
            
            # Get action from agent
            legal_moves = env.generate_all_moves(env.to_move)
            if not legal_moves:
                break
                
            chosen_move, chosen_action = agent.get_action(env, legal_moves, training=True)
            if chosen_move is None:
                break
            
            # Apply move and get RAW reward
            raw_reward = env.apply_move(chosen_move)
            total_raw_reward += raw_reward
            
            # === NORMALIZE REWARD ===
            normalized_reward = agent.normalize_reward(raw_reward)
            total_normalized_reward += normalized_reward
            
            # Get next state
            next_state = env.copy()
            done = env.game_over()
            
            # Update Q-value with NORMALIZED reward
            agent.update_q_value(current_state, chosen_action, normalized_reward, next_state, done)
            
            # Update model with NORMALIZED reward
            agent.update_model(current_state, chosen_action, next_state, normalized_reward)
            
            # Perform planning steps
            agent.plan()
            
            move_count += 1
        
        # Game finished - get final results
        final_scores, winner = env.final_scores_and_winner()
        win_counts[winner] += 1
        
        episode_rewards.append(total_normalized_reward)
        episode_lengths.append(move_count)
        
        # Decay epsilon for exploration
        if episode % 100 == 0:
            agent.epsilon = max(0.01, agent.epsilon * 0.95)
        
        # Logging and progress
        if episode % 50 == 0:
            avg_norm_reward = np.mean(episode_rewards[-50:])
            avg_length = np.mean(episode_lengths[-50:])
            recent_win_rate = win_counts[1] / episode
            
            # TensorBoard logging
            writer.add_scalar("Training/Average_Normalized_Reward", avg_norm_reward, episode)
            writer.add_scalar("Training/Average_Raw_Reward", total_raw_reward / move_count if move_count > 0 else 0, episode)
            writer.add_scalar("Training/Average_Length", avg_length, episode)
            writer.add_scalar("Training/Win_Rate_Player1", recent_win_rate, episode)
            writer.add_scalar("Training/Epsilon", agent.epsilon, episode)
            writer.add_scalar("Training/Q_Table_Size", len(agent.Q), episode)
            writer.add_scalar("Training/Model_Size", len(agent.visited_sa), episode)
            writer.add_scalar("Training/Reward_Mean", agent.running_mean, episode)
            writer.add_scalar("Training/Reward_Std", agent.running_std, episode)
            
            print(f"Episode {episode}/{num_episodes} | "
                  f"Norm R: {avg_norm_reward:.3f} | "
                  f"Raw R: {total_raw_reward/move_count:.3f} | "
                  f"Len: {avg_length:.1f} | "
                  f"Win P1: {recent_win_rate:.3f} | "
                  f"ε: {agent.epsilon:.3f} | "
                  f"Q: {len(agent.Q)} | "
                  f"μ={agent.running_mean:.2f}, σ={agent.running_std:.2f}")
    
    writer.close()
    print("🎯 Dyna-Q training complete!")
    return agent

In [22]:
# ----------------------- Evaluation Function ------------------------
def evaluate_dyna_q_agent(agent, env_factory, num_games=10, verbose=False):
    """Evaluate the Dyna-Q agent's performance"""
    win_counts = {1: 0, -1: 0, 0: 0}
    game_lengths = []
    score_diffs = []
    
    print(f"🔍 Evaluating Dyna-Q agent over {num_games} games...")
    
    for game in range(num_games):
        env = env_factory()
        move_count = 0
        
        if verbose:
            print(f"\n🎮 Game {game + 1}")
            
        while not env.game_over() and move_count < 300:
            legal_moves = env.generate_all_moves(env.to_move)
            if not legal_moves:
                break
                
            # Use agent for both players (self-play evaluation)
            chosen_move, _ = agent.get_action(env, legal_moves, training=False)
            if chosen_move is None:
                break
                
            if verbose and move_count < 10:  # Show first few moves
                piece = env.pieces.get(chosen_move.path[0], None)
                if piece:
                    piece_type = "Dama" if piece.dama else "Regular"
                    start_pos = chosen_move.path[0]
                    end_pos = chosen_move.path[-1]
                    print(f"  Player {env.to_move}: {piece_type} {piece.value:+d} "
                          f"({start_pos[1]}, {start_pos[0]}) -> ({end_pos[1]}, {end_pos[0]}) "
                          f"| Score gain: {chosen_move.score_gain:+.1f}")
            
            env.apply_move(chosen_move)
            move_count += 1
        
        final_scores, winner = env.final_scores_and_winner()
        win_counts[winner] += 1
        game_lengths.append(move_count)
        score_diffs.append(final_scores[1] - final_scores[-1])
        
        if verbose:
            print(f"  Winner: Player {winner}, Scores: {final_scores}, Length: {move_count}")
    
    # Calculate statistics
    avg_length = np.mean(game_lengths)
    avg_score_diff = np.mean(score_diffs)
    win_rate_p1 = win_counts[1] / num_games
    win_rate_p2 = win_counts[-1] / num_games
    draw_rate = win_counts[0] / num_games
    
    print(f"\n📊 Evaluation Results:")
    print(f"  Win Rate Player 1: {win_rate_p1:.3f}")
    print(f"  Win Rate Player 2: {win_rate_p2:.3f}")
    print(f"  Draw Rate: {draw_rate:.3f}")
    print(f"  Average Game Length: {avg_length:.1f}")
    print(f"  Average Score Difference: {avg_score_diff:.1f}")
    print(f"  Q-table Size: {len(agent.Q)} states")
    print(f"  Model Size: {len(agent.visited_sa)} state-action pairs")
    
    return {
        'win_rates': {'p1': win_rate_p1, 'p2': win_rate_p2, 'draw': draw_rate},
        'avg_length': avg_length,
        'avg_score_diff': avg_score_diff,
        'q_table_size': len(agent.Q),
        'model_size': len(agent.visited_sa)
    }

In [23]:
# ----------------------- Usage Example ------------------------
if __name__ == "__main__":
    print("🚀 Starting Dyna-Q Training for Integer Damath...")

🚀 Starting Dyna-Q Training for Integer Damath...


In [24]:
# ----------------------- Demo Game Function ------------------------
def demo_dyna_q_game(agent, env_factory, verbose=True):
    """Run a demonstration game showing the agent's decision-making"""
    env = env_factory()
    move_count = 0
    
    print("🎭 Dyna-Q Agent Demonstration Game")
    print("=" * 50)
    
    if verbose:
        env.print_board()
        print(f"Initial scores: {env.scores}")
    
    while not env.game_over() and move_count < 50:  # Limit demo to 50 moves
        if verbose:
            print(f"\n--- Move {move_count + 1} ---")
            print(f"Player {env.to_move}'s turn")
        
        legal_moves = env.generate_all_moves(env.to_move)
        if not legal_moves:
            break
        
        # Get agent's action with Q-values for top moves
        state_hash = state_to_hash(env)
        action_ids, moves = get_legal_actions(env)
        
        if verbose and len(action_ids) > 1:
            # Show Q-values for available actions
            q_values = [(aid, agent.Q[state_hash][aid]) for aid in action_ids]
            q_values.sort(key=lambda x: x[1], reverse=True)
            
            print(f"Top 3 Q-values:")
            for i, (aid, qval) in enumerate(q_values[:3]):
                print(f"  {i+1}. {aid} -> Q = {qval:.3f}")
        
        chosen_move, chosen_action = agent.get_action(env, legal_moves, training=False)
        if chosen_move is None:
            break
        
        # Show chosen move details
        if verbose:
            piece = env.pieces.get(chosen_move.path[0], None)
            if piece:
                piece_type = "Dama" if piece.dama else "Regular"
                start_pos = chosen_move.path[0]
                end_pos = chosen_move.path[-1]
                print(f"Chosen: {piece_type} {piece.value:+d} "
                      f"({start_pos[1]}, {start_pos[0]}) -> ({end_pos[1]}, {end_pos[0]})")
                print(f"Score gain: {chosen_move.score_gain:+.1f}")
                print(f"Q-value: {agent.Q[state_hash][chosen_action]:.3f}")
        
        env.apply_move(chosen_move, verbose=False)
        move_count += 1
        
        if verbose and move_count % 5 == 0:
            env.print_board()
            print(f"Current scores: {env.scores}")
    
    # Final results
    if verbose:
        env.print_board()
    
    final_scores, winner = env.final_scores_and_winner()
    print(f"\n🏁 Demo Game Complete!")
    print(f"Final scores: {final_scores}")
    print(f"Winner: Player {winner}")
    print(f"Total moves: {move_count}")
    
    return final_scores, winner, move_count

# ----------------------- Hyperparameters ------------------------
ALPHA = 0.01          # Learning rate
GAMMA = 0.9         # Discount factor
EPSILON = 0.2        # Initial exploration rate
PLANNING_STEPS = 50  # Number of planning steps per real step
NUM_EPISODES = 200  # Number of training episodes
MAX_MOVES_PER_GAME = 100

# ----------------------- Environment Setup ------------------------
operator_pattern_official = [
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '/', '/', '-', '-', '/', '+', 'x']
]

def env_factory():
    e = DamathEnv(operator_pattern=operator_pattern_official)
    e.init_default_integer_setup()
    return e

# ----------------------- Initialize and Train Dyna-Q Agent ------------------------
print("🚀 Initializing Dyna-Q Agent for Integer Damath...")
dyna_q_agent = DynaQAgent(alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, planning_steps=PLANNING_STEPS)

print("🎲 Starting Dyna-Q training...")
trained_agent = train_dyna_q_agent(env_factory, dyna_q_agent, num_episodes=NUM_EPISODES, max_moves_per_game=MAX_MOVES_PER_GAME)

# Save the trained agent
agent_save_path = f"dyna_q_agent_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
trained_agent.save_agent(agent_save_path)
print(f"💾 Trained agent saved to: {agent_save_path}")

# Evaluate the trained agent
print("\n" + "="*60)
print("🔍 EVALUATING TRAINED DYNA-Q AGENT")
print("="*60)

evaluation_results = evaluate_dyna_q_agent(trained_agent, env_factory, num_games=200, verbose=False)

# Run a demonstration game
print("\n" + "="*60)
print("🎭 DEMONSTRATION GAME")
print("="*60)
demo_results = demo_dyna_q_game(trained_agent, env_factory, verbose=True)

print("\n✅ All tasks completed successfully!")

🚀 Initializing Dyna-Q Agent for Integer Damath...
🎲 Starting Dyna-Q training...
📈 Logging Dyna-Q training to TensorBoard: runs/damath_dynaq_20251011_061315
Episode 50/200 | Norm R: -1.520 | Raw R: -0.047 | Len: 46.1 | Win P1: 0.340 | ε: 0.200 | Q: 2148 | μ=-0.00, σ=0.31
Episode 100/200 | Norm R: 0.186 | Raw R: 0.049 | Len: 48.0 | Win P1: 0.450 | ε: 0.190 | Q: 4152 | μ=0.01, σ=0.31
Episode 150/200 | Norm R: -0.658 | Raw R: -0.010 | Len: 48.1 | Win P1: 0.453 | ε: 0.190 | Q: 6103 | μ=0.02, σ=0.31
Episode 200/200 | Norm R: -0.602 | Raw R: 0.062 | Len: 48.9 | Win P1: 0.445 | ε: 0.180 | Q: 8077 | μ=0.03, σ=0.32
🎯 Dyna-Q training complete!
✅ Agent saved successfully to dyna_q_agent_20251011_061455.pkl
💾 Trained agent saved to: dyna_q_agent_20251011_061455.pkl

🔍 EVALUATING TRAINED DYNA-Q AGENT
🔍 Evaluating Dyna-Q agent over 200 games...

📊 Evaluation Results:
  Win Rate Player 1: 0.555
  Win Rate Player 2: 0.440
  Draw Rate: 0.005
  Average Game Length: 51.0
  Average Score Difference: 17.9
 

In [25]:
def simulate_dyna_q_game_minimal_all_boards(agent, env_factory, max_moves=300):
    """Minimal simulation showing all boards"""
    env = env_factory()
    move_count = 0
    
    print("🎮 Dyna-Q Game - All Boards\n")
    print("Initial Board:")
    env.print_board()
    print(f"Scores: {env.scores}\n")
    
    while not env.game_over() and move_count < max_moves:
        legal_moves = env.generate_all_moves(env.to_move)
        if not legal_moves:
            break
        
        chosen_move, _ = agent.get_action(env, legal_moves, training=False)
        if chosen_move is None:
            break
        
        piece = env.pieces.get(chosen_move.path[0])
        start, end = chosen_move.path[0], chosen_move.path[-1]
        
        print(f"\n{'='*60}")
        print(f"Move {move_count + 1}: Player {env.to_move}")
        if piece:
            piece_type = "Dama" if piece.dama else "Reg"
            print(f"{piece_type} {piece.value:+d}: (x={start[1]},y={start[0]}) -> (x={end[1]},y={end[0]}) | ΔScore={chosen_move.score_gain:+.1f}")
        
        env.apply_move(chosen_move, verbose=False)
        move_count += 1
        
        env.print_board()
        print(f"Scores: Blue={env.scores[1]:.1f}, Red={env.scores[-1]:.1f}")
    
    final_scores, winner = env.final_scores_and_winner()
    print(f"\n🏁 Game Over! Winner: Player {winner}")
    print(f"Final: Blue={final_scores[1]:.1f}, Red={final_scores[-1]:.1f}")
    print(f"Total moves: {move_count}")
    
    return final_scores, winner, move_count

In [26]:
# ----------------------- Demo Game Function ------------------------
def demo_dyna_q_game(agent, env_factory, verbose=True):
    """Run a demonstration game showing the agent's decision-making"""
    env = env_factory()
    move_count = 0
    
    print("🎭 Dyna-Q Agent Demonstration Game")
    print("=" * 50)
    
    if verbose:
        env.print_board()
        print(f"Initial scores: {env.scores}")
    
    while not env.game_over() and move_count < 50:  # Limit demo to 50 moves
        if verbose:
            print(f"\n--- Move {move_count + 1} ---")
            print(f"Player {env.to_move}'s turn")
        
        legal_moves = env.generate_all_moves(env.to_move)
        if not legal_moves:
            break
        
        # Get agent's action with Q-values for top moves
        state_hash = state_to_hash(env)
        action_ids, moves = get_legal_actions(env)
        
        if verbose and len(action_ids) > 1:
            # Show Q-values for available actions
            q_values = [(aid, agent.Q[state_hash][aid]) for aid in action_ids]
            q_values.sort(key=lambda x: x[1], reverse=True)
            
            print(f"Top 3 Q-values:")
            for i, (aid, qval) in enumerate(q_values[:3]):
                print(f"  {i+1}. {aid} -> Q = {qval:.3f}")
        
        chosen_move, chosen_action = agent.get_action(env, legal_moves, training=False)
        if chosen_move is None:
            break
        
        # Show chosen move details
        if verbose:
            piece = env.pieces.get(chosen_move.path[0], None)
            if piece:
                piece_type = "Dama" if piece.dama else "Regular"
                start_pos = chosen_move.path[0]
                end_pos = chosen_move.path[-1]
                print(f"Chosen: {piece_type} {piece.value:+d} "
                      f"({start_pos[1]}, {start_pos[0]}) -> ({end_pos[1]}, {end_pos[0]})")
                print(f"Score gain: {chosen_move.score_gain:+.1f}")
                print(f"Q-value: {agent.Q[state_hash][chosen_action]:.3f}")
        
        env.apply_move(chosen_move, verbose=False)
        move_count += 1
        
        if verbose and move_count % 5 == 0:
            env.print_board()
            print(f"Current scores: {env.scores}")
    
    # Final results
    if verbose:
        env.print_board()
    
    final_scores, winner = env.final_scores_and_winner()
    print(f"\n🏁 Demo Game Complete!")
    print(f"Final scores: {final_scores}")
    print(f"Winner: Player {winner}")
    print(f"Total moves: {move_count}")
    
    return final_scores, winner, move_count

In [27]:
simulate_dyna_q_game_minimal_all_boards(trained_agent, env_factory, max_moves=50)


🎮 Dyna-Q Game - All Boards

Initial Board:

        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    +   ##    -   ##    /   ##    x |  4
 3 |    x   ##    /   ##    -   ##    +   ## |  3
 2 |   ##  🔵-9   ##  🔵06   ##  🔵-1   ##  🔵04 |  2
 1 |  🔵00   ##  🔵-3   ##  🔵10   ##  🔵-7   ## |  1
 0 |   ## 🔵-11   ##  🔵08   ##  🔵-5   ##  🔵02 |  0
     --------------------------------
        0    1    2    3    4    5    6    7
Scores: {1: 0.0, -1: 0.0}


Move 1: Player 1
Reg -9: (x=1,y=2) -> (x=2,y=3) | ΔScore=+0.0

        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    +   ##    -   ##    /   ##    x |  4
 3 |    x   ##  

({1: -53.0, -1: -21.0}, -1, 50)